<a href="https://colab.research.google.com/github/felicity8899/agent-trading/blob/main/homework1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)

In [10]:
tables = pd.read_html(response.text)

/tmp/ipykernel_1936/3072015354.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [13]:
df = tables[0]
df

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [29]:
df['Year Added'] = pd.to_datetime(df['Date added'], errors='coerce').dt.year
df_since_2020 = df[df['Year Added'] >= 2020]
df.head(5)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,Year Added
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,1957
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,2017
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,1957
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888),2012
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989,2011


In [34]:
counts = df_since_2020['Year Added'].value_counts().sort_index()

print("2020年以来每年新增数量统计：")
print(counts)

# 5. 直接找出数量最多的一年
max_year = counts.idxmax()
max_count = counts.max()
print(f"\n其中新增最多的是 {int(max_year)} 年，共新增了 {max_count} 家公司。")

2020年以来每年新增数量统计：
Year Added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64

其中新增最多的是 2025 年，共新增了 18 家公司。


In [35]:
import pandas as pd
import yfinance as yf

# Dictionary mapping country/index names to Yahoo Finance tickers
tickers = {
    "United States (S&P 500)": "^GSPC",
    "China (Shanghai Composite)": "000001.SS",
    "Hong Kong (Hang Seng Index)": "^HSI",
    "Australia (S&P/ASX 200)": "^AXJO",
    "India (Nifty 50)": "^NSEI",
    "Canada (S&P/TSX Composite)": "^GSPTSE",
    "Germany (DAX)": "^GDAXI",
    "United Kingdom (FTSE 100)": "^FTSE",
    "Japan (Nikkei 225)": "^N225",
    "Mexico (IPC Mexico)": "^MXX",
    "Brazil (Ibovespa)": "^BVSP",
}

start_date = "2026-01-01"
end_date = "2026-08-21"

In [36]:
performance_data = []
for name, ticker in tickers.items():
  # Download daily historical data
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)

  if not df.empty:
    # Handle multi-index columns in newer yfinance versions if needed
    close_col = (
        df["Close"][ticker]
        if isinstance(df.columns, pd.MultiIndex)
        else df["Close"]
    )
    close_col = close_col.dropna()

    if len(close_col) >= 2:
      start_price = close_col.iloc[0]
      end_price = close_col.iloc[-1]
      ytd_return = ((end_price - start_price) / start_price) * 100
      performance_data.append({
          "Index / Country": name,
          "Ticker": ticker,
          "Start Price": start_price,
          "End Price (Aug 21)": end_price,
          "YTD Return (%)": ytd_return,
      })

# Convert to DataFrame and sort by performance
perf_df = pd.DataFrame(performance_data)
perf_df = perf_df.sort_values(by="YTD Return (%)", ascending=False).reset_index(
    drop=True
)
print(perf_df.to_string(index=False))

/tmp/ipykernel_1936/2073818766.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1936/2073818766.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1936/2073818766.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1936/2073818766.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1936/2073818766.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1936/2073818766.py:4: FutureWarning

            Index / Country    Ticker   Start Price  End Price (Aug 21)  YTD Return (%)
         Japan (Nikkei 225)     ^N225  51832.800781        66216.789062       27.750745
 Canada (S&P/TSX Composite)   ^GSPTSE  31883.400391        36365.398438       14.057466
    United States (S&P 500)     ^GSPC   6858.470215         7641.160156       11.412019
  United Kingdom (FTSE 100)     ^FTSE   9951.099609        10748.200195        8.010176
              Germany (DAX)    ^GDAXI  24539.339844        25983.039062        5.883203
          Brazil (Ibovespa)     ^BVSP 160539.000000       167927.000000        4.601997
    Australia (S&P/ASX 200)     ^AXJO   8727.799805         9083.799805        4.078920
        Mexico (IPC Mexico)      ^MXX  64141.359375        64349.800781        0.324972
Hong Kong (Hang Seng Index)      ^HSI  26338.470703        25698.490234       -2.429832
 China (Shanghai Composite) 000001.SS   4023.416992         3903.720947       -2.974985
           India (Nifty 50)     

In [37]:
import numpy as np
import pandas as pd
import yfinance as yf

# 1. Download S&P 500 historical daily data from 1950 to present
sp500 = yf.download("^GSPC", start="1950-01-01", progress=False)
prices = sp500["Close"]

# If columns are multi-index (common in newer yfinance versions), flatten them
if isinstance(prices, pd.DataFrame):
  prices = prices.iloc[:, 0]

# 2. Identify all-time high closing price points
running_max = prices.cummax()
is_ath = prices == running_max

# 3. Find drawdowns between consecutive all-time highs
# Find periods between ATHs and calculate the minimum price (trough) in between
drawdowns = []
ath_dates = prices[is_ath].index

for i in range(len(ath_dates) - 1):
  start_date = ath_dates[i]
  end_date = ath_dates[i + 1]

  peak_price = prices[start_date]
  period_data = prices.loc[start_date:end_date]
  trough_price = period_data.min()
  trough_date = period_data.idxmin()

  drawdown_pct = ((peak_price - trough_price) / peak_price) * 100
  duration_days = (trough_date - start_date).days

  drawdowns.append({
      "Peak Date": start_date,
      "Trough Date": trough_date,
      "Peak Price": peak_price,
      "Trough Price": trough_price,
      "Drawdown (%)": drawdown_pct,
      "Duration (Days)": duration_days,
  })

df_drawdowns = pd.DataFrame(drawdowns)

# 4. Filter for corrections with at least 5% drawdown
df_corrections = df_drawdowns[df_drawdowns["Drawdown (%)"] >= 5.0]

# 5. Determine percentiles (25th, 50th/median, 75th) for drawdowns and durations
drawdown_percentiles = np.percentile(
    df_corrections["Drawdown (%)"], [25, 50, 75]
)
duration_percentiles = np.percentile(
    df_corrections["Duration (Days)"], [25, 50, 75]
)

print(
    "Drawdown Percentiles (25th, 50th, 75th):",
    drawdown_percentiles,
)
print(
    "Duration Days Percentiles (25th, 50th, 75th):",
    duration_percentiles,
)

/tmp/ipykernel_1936/2048690268.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download("^GSPC", start="1950-01-01", progress=False)


Drawdown Percentiles (25th, 50th, 75th): [ 6.23467673  7.98635756 14.01982585]
Duration Days Percentiles (25th, 50th, 75th): [22.   40.5  86.25]


In [45]:
import numpy as np
import pandas as pd
import yfinance as yf

# 1. Load earnings dates for AMZN
ticker = "AMZN"
ticker_obj = yf.Ticker(ticker)
earnings_df = ticker_obj.get_earnings_dates()

# Clean and drop rows where Surprise % or Reported EPS is missing (e.g., future dates)
earnings_df = earnings_df.dropna(subset=["Surprise(%)", "Reported EPS"])

# Ensure index is timezone-naive datetime to match price history
earnings_df.index = pd.to_datetime(earnings_df.index).tz_localize(None)

# 2. Download historical daily stock price data
# Start a bit earlier than the first earnings date to ensure 3-day windows are available
start_date = earnings_df.index.min() - pd.Timedelta(days=10)
hist_df = yf.download(
    ticker, start=start_date, progress=False
)  # handles recent history

# Flatten multi-index columns if present in newer yfinance versions
if isinstance(hist_df.columns, pd.MultiIndex):
  hist_df = hist_df.xs("Close", level=0, axis=1)
  # If it returns a DataFrame with one column, convert to Series or select series
  if isinstance(hist_df, pd.DataFrame):
    close_prices = hist_df.iloc[:, 0]
else:
  close_prices = hist_df["Close"]

# Ensure close_prices index is timezone-naive
close_prices.index = pd.to_datetime(close_prices.index).tz_localize(None)


# 3. Calculate 2-day percentage changes for all historical trading days
# 3 consecutive trading days: Day 1 (t-1), Day 2 (t, earnings day), Day 3 (t+1)
# Return = Close_Day3 / Close_Day1 - 1
# Shift(-1) brings Day 3 to Day 1's row, Shift(1) brings Day 1 to Day 1's row...
# Alternatively, using rolling or direct index mapping:
returns_2day = []
matched_surprises = []

for e_date, row in earnings_df.iterrows():
  # Find the trading day closest to or matching the earnings date
  # In yfinance get_earnings_dates, the index represents the announcement date/time
  trading_days = close_prices.index

  # Get position of the earnings date or the nearest trading day
  sub_days = trading_days[trading_days >= e_date]
  if len(sub_days) > 0:
    announcement_idx = trading_days.get_loc(sub_days[0])

    # Ensure we have at least 1 day before and 1 day after
    if announcement_idx > 0 and announcement_idx < len(trading_days) - 1:
      day1_price = close_prices.iloc[announcement_idx - 1]  # Day 1 (t-1)
      day3_price = close_prices.iloc[announcement_idx + 1]  # Day 3 (t+1)

      ret_2d = (day3_price / day1_price) - 1
      returns_2day.append(ret_2d)
      matched_surprises.append(row["Surprise(%)"])

# Create analysis dataframe
analysis_df = pd.DataFrame({
    "Return_2D": returns_2day,
    "Surprise_Pct": matched_surprises,
})

# 4. Filter for positive earnings surprises
positive_surprises_df = analysis_df[analysis_df["Surprise_Pct"] > 0]

# Calculate the median 2-day return
median_return = positive_surprises_df["Return_2D"].median() * 100  # in percentage

# Calculate correlation between the 2-day stock return and earnings surprise magnitude
correlation = positive_surprises_df["Return_2D"].corr(
    positive_surprises_df["Surprise_Pct"]
)

print(f"Number of positive surprise events: {len(positive_surprises_df)}")
print(f"Median 2-day percentage change: {median_return:.2f}%")
print(f"Correlation with surprise magnitude: {correlation:.4f}")

Number of positive surprise events: 20
Median 2-day percentage change: -1.74%
Correlation with surprise magnitude: 0.4721


/tmp/ipykernel_1936/4166468000.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  hist_df = yf.download(
